# Population-level analyses notebook

Written by WMS to perform various population-level analyses on 1P calcium imaging data

last edit 2/28/26 wms

### Imports and data structure construction

In [ ]:
import pandas as pd
import numpy as np
import os
import pickle
import utils
import decoding
import scipy.io
import scipy.stats as stats
from scipy.ndimage import gaussian_filter1d
from scipy.spatial.distance import euclidean
import seaborn as sns
import matplotlib.pyplot as plt
from pca import pca
from natsort import natsorted
from sklearn import svm
from sklearn import metrics
from sklearn.svm import LinearSVC
#from sklearn.preprocessing import StandardScaler

In [ ]:
datapath = r"C:\miniscope_files\analyzed\6175\2024_02_29\13_37_01\bento\analysis_results"

prep_path = os.path.join(datapath, "data_prep")

create directory to save results into

In [ ]:
poppath = os.path.join(datapath,"population_results")

In [ ]:
#make the dir
if not os.path.isdir(poppath):
    os.mkdir(poppath)

Import aligned timestamps file and get approximate frame rate

In [ ]:
timestamp_file = os.path.join(prep_path, "aligned_timestamps.csv")
timestamps = pd.read_csv(timestamp_file, header=0)
timestamps

In [ ]:
approx_fr = utils.approx_fr(timestamps)
approx_fr

Load in z-scored dF/F data from session

In [ ]:
dfz_file = os.path.join(prep_path, "zscored_dff_filtered.csv")

dff_Z = pd.read_csv(dfz_file, header=0)

#also define the number of cells in your dataset
numcells = dff_Z.shape[0]
print(numcells)

In [ ]:
dff_Z

In [ ]:
if timestamps.shape[0] == dff_Z.shape[1]:
    print('aligned timestamps and z-scored dF/F data have the same number of frames')
else:
    print('aligned timestamps and z-scored dF/F data do NOT have the same number of frames -- investigate discrepancies')

In [ ]:
#load in discritized events (S data)
sdata_file = os.path.join(prep_path, "sdata_filtered.csv")

sData = pd.read_csv(sdata_file, header=0)

In [ ]:
sData

In [ ]:
#SdZ = sData.apply(stats.zscore, axis=1, result_type='broadcast')

In [ ]:
#SdZ

Load in behavioral raster from session (including partner and novel interaction frames)

In [ ]:
rast_file = os.path.join(prep_path, "aligned_rasters.csv")

beh_rast = pd.read_csv(rast_file, index_col=0,header=0)

beh_rast

In [ ]:
if beh_rast.shape[1] == dff_Z.shape[1]:
    print('behavior rasters and z-scored dF/F data have the same number of frames')
else:
    print('behavior rasters and z-scored dF/F data do NOT have the same number of frames -- investigate discrepancies')

In [ ]:
#confirming if partner and novel sides were assigned correctly
beh_rast.loc[:,beh_rast.loc['novel_active']==1]

Import active interaction labels

In [ ]:
#partner and novel
act_pint_file = os.path.join(prep_path, "active_partner_interactions.csv")
act_pints = pd.read_csv(act_pint_file, index_col=0,header=0)

act_nint_file = os.path.join(prep_path, "active_novel_interactions.csv")
act_nints = pd.read_csv(act_nint_file, index_col=0,header=0)

#left and right, both social trials
act_rint_social_file = os.path.join(prep_path, "active_rightcup_interactions_socialtrials.csv")
act_rints_social = pd.read_csv(act_rint_social_file, index_col=0,header=0)

act_lint_social_file = os.path.join(prep_path, "active_leftcup_interactions_socialtrials.csv")
act_lints_social = pd.read_csv(act_lint_social_file, index_col=0,header=0)

#left and right, empty1
act_rint_empty1_file = os.path.join(prep_path, "active_rightcup_interactions_empty1.csv")
act_rints_empty1 = pd.read_csv(act_rint_empty1_file, index_col=0,header=0)

act_lint_empty1_file = os.path.join(prep_path, "active_leftcup_interactions_empty1.csv")
act_lints_empty1 = pd.read_csv(act_lint_empty1_file, index_col=0,header=0)

In [ ]:
act_pints

Get the four identity/location interaction subclasses

In [ ]:
pr_ints = utils.get_bout_subtypes(act_rints_social,act_pints,label='p_r',length_min=0) #changing length_min=0 because I am going to merge first before dropping anything less than a second, now
pl_ints = utils.get_bout_subtypes(act_lints_social,act_pints,label='p_l',length_min=0)

nr_ints = utils.get_bout_subtypes(act_rints_social,act_nints,label='n_r',length_min=0)
nl_ints = utils.get_bout_subtypes(act_lints_social,act_nints,label='n_l',length_min=0)

In [ ]:
pl_ints

Assign subclasses to empty left and empty right

In [ ]:
act_rints_empty1['type']='r_e1'
act_lints_empty1['type']='l_e1'

In [ ]:
act_rints_empty1

Merge bouts of the same type that are less than 1 second from each other

In [ ]:
#first determine whether there are any dropped scope frames
#if so, avoid merging bouts that were originally split because there were dropped frames between them  (this is built into merge_close_bouts)
dropped_frame_info = utils.get_dropped_frame_info(dff_Z)
print(f"Found {len(dropped_frame_info['dropped_ranges'])} dropped frame intervals: {dropped_frame_info['dropped_ranges']}")

In [ ]:
merge_thresh_sec=1.0

nl_ints_merged, n_merges_nl, n_skips_nl = utils.merge_close_bouts(nl_ints,
                                                                  threshold_sec=merge_thresh_sec,
                                                                  fr=approx_fr,
                                                                  dropped_frame_info=dropped_frame_info)
print(f'number of merged novel/left interactions: {n_merges_nl}; number skipped due to dropped frames: {n_skips_nl}')

nr_ints_merged, n_merges_nr, n_skips_nr = utils.merge_close_bouts(nr_ints,
                                                                  threshold_sec=merge_thresh_sec,
                                                                  fr=approx_fr,
                                                                  dropped_frame_info=dropped_frame_info)
print(f'number of merged novel/right interactions: {n_merges_nr}; number skipped due to dropped frames: {n_skips_nr}')

pl_ints_merged, n_merges_pl, n_skips_pl = utils.merge_close_bouts(pl_ints,
                                                                  threshold_sec=merge_thresh_sec,
                                                                  fr=approx_fr,
                                                                  dropped_frame_info=dropped_frame_info)
print(f'number of merged partner/left interactions, social frames: {n_merges_pl}; number skipped due to dropped frames: {n_skips_pl}')

pr_ints_merged, n_merges_pr, n_skips_pr = utils.merge_close_bouts(pr_ints,
                                                                  threshold_sec=merge_thresh_sec,
                                                                  fr=approx_fr,
                                                                  dropped_frame_info=dropped_frame_info)
print(f'number of merged partner/right interactions, social frames: {n_merges_pr}; number skipped due to dropped frames: {n_skips_pr}')

act_lints_merged_empty1, n_merges_lints_empty1, n_skips_lints_empty1 = utils.merge_close_bouts(act_lints_empty1,
                                                                                         threshold_sec=merge_thresh_sec,
                                                                                         fr=approx_fr,
                                                                                         dropped_frame_info=dropped_frame_info)
print(f'number of merged active left interactions, empty1: {n_merges_lints_empty1}; number skipped due to dropped frames: {n_skips_lints_empty1}')

act_rints_merged_empty1, n_merges_rints_empty1, n_skips_rints_empty1 = utils.merge_close_bouts(act_rints_empty1,
                                                                                         threshold_sec=merge_thresh_sec,
                                                                                         fr=approx_fr,
                                                                                         dropped_frame_info=dropped_frame_info)
print(f'number of merged active right interactions, empty1: {n_merges_rints_empty1}; number skipped due to dropped frames: {n_skips_rints_empty1}')

In [ ]:
nr_ints

In [ ]:
pl_ints_merged

Drop all bouts that are not at least one second long (for pseudotrial construction)

In [ ]:
def drop_short_bouts(ints_df, min_bout_length):
    if len(ints_df) > 0:
        ints_df_long_bouts = ints_df[ints_df['length']>=min_bout_length].copy().reset_index(drop=True)
        return ints_df_long_bouts
    else:
        return ints_df.copy()

In [ ]:
min_bout_length = 1 #seconds

In [ ]:
pr_ints_merged = drop_short_bouts(pr_ints_merged,min_bout_length)
pl_ints_merged = drop_short_bouts(pl_ints_merged,min_bout_length)
nr_ints_merged = drop_short_bouts(nr_ints_merged,min_bout_length)
nl_ints_merged = drop_short_bouts(nl_ints_merged,min_bout_length)
act_lints_merged_empty1 = drop_short_bouts(act_lints_merged_empty1,min_bout_length)
act_rints_merged_empty1 = drop_short_bouts(act_rints_merged_empty1,min_bout_length)

In [ ]:
pl_ints_merged

In [ ]:
act_lints_merged_empty1

In [ ]:
for ints_df in [nl_ints_merged,nr_ints_merged,pl_ints_merged,pr_ints_merged]:
    if len(ints_df)>0:
        utils.check_inclusion_threshold(ints_df,min_int_number=2)
    else:
        print(f"WARNING: at least one df has no interactions")

In [ ]:
for ints_df in [act_rints_merged_empty1,act_lints_merged_empty1]:
    if len(ints_df)>0:
        utils.check_inclusion_threshold(ints_df,min_int_number=2)
    else:
        print(f"WARNING: at least one df has no interactions")

Extract the neural data for each bout of a given interaction type and calculate that interaction bout's mean population vector

In [ ]:
pr_dffs, pr_pvs = utils.traces_pv_by_bout(pr_ints_merged,dff_Z)

In [ ]:
pl_dffs, pl_pvs = utils.traces_pv_by_bout(pl_ints_merged,dff_Z)

In [ ]:
nl_dffs, nl_pvs = utils.traces_pv_by_bout(nl_ints_merged,dff_Z)

In [ ]:
nr_dffs, nr_pvs = utils.traces_pv_by_bout(nr_ints_merged,dff_Z)

In [ ]:
l_e1_dffs, l_e1_pvs = utils.traces_pv_by_bout(act_lints_merged_empty1,dff_Z)

In [ ]:
r_e1_dffs, r_e1_pvs = utils.traces_pv_by_bout(act_rints_merged_empty1,dff_Z)

Construct 1 sec-long pseudotrials for Mahalanobis distance calculation and decoding

In [ ]:
bin_size = 1 #seconds

In [ ]:
pr_pts = utils.make_pseudotrials(pr_ints_merged,approx_fr,entire=True,length=bin_size)
_ = utils.validate_pseudotrials(pr_pts,dff_Z)

In [ ]:
pl_pts = utils.make_pseudotrials(pl_ints_merged,approx_fr,entire=True,length=bin_size)
_ = utils.validate_pseudotrials(pl_pts,dff_Z)

In [ ]:
nl_pts = utils.make_pseudotrials(nl_ints_merged,approx_fr,entire=True,length=bin_size)
_ = utils.validate_pseudotrials(nl_pts,dff_Z)

In [ ]:
nr_pts = utils.make_pseudotrials(nr_ints_merged,approx_fr,entire=True,length=bin_size)
_ = utils.validate_pseudotrials(nr_pts,dff_Z)

In [ ]:
l_e1_pts = utils.make_pseudotrials(act_lints_merged_empty1,approx_fr,entire=True,length=bin_size)
_ = utils.validate_pseudotrials(l_e1_pts,dff_Z)

In [ ]:
r_e1_pts = utils.make_pseudotrials(act_rints_merged_empty1,approx_fr,entire=True,length=bin_size)
_ = utils.validate_pseudotrials(r_e1_pts,dff_Z)

In [ ]:
nl_ints_merged

In [ ]:
pr_pts

Quality control measure: how sparse is your activity and how many 'empty' population vectors do you have with the chosen bin size?

In [ ]:
#create summary dictionary to save out metrics re the sparsity of activity
sparsity_dict = {}

In [ ]:
event_rates, median_rate, mean_rate = utils.get_event_rates(sData, approx_fr)
bins_per_event = 1 / (median_rate * bin_size)
neurons_per_bin = numcells * median_rate * bin_size
events_per_bin = mean_rate * bin_size

range_summary = f"Range: {event_rates.min():.3f} - {event_rates.max():.3f} events/sec"

sparsity_dict['all_event_rates']=event_rates
sparsity_dict['median_rate']=median_rate
sparsity_dict['mean_rate']=mean_rate
sparsity_dict['event_rate_range']=range_summary

print(f"Median event rate: {median_rate:.3f} events/sec ~= each neuron active approximately once every {bins_per_event:.1f} bins ~= approximately {neurons_per_bin:.1f} neurons active per bin")
print(f"Mean event rate: {mean_rate:.3f} events/sec = {events_per_bin:.2f} events/bin per neuron")
print(range_summary)

In [ ]:
def get_empty_pseudotrial_props(pts_df,sData,buffer_frames=0,verbose=True):
    pt_event_sums, pt_n_empty, pt_empty_indices = utils.check_empty_pseudotrials(pts_df,sData,buffer_frames=buffer_frames,verbose=verbose)
    prop_pts_empty = pt_n_empty/len(pts_df)
    return prop_pts_empty

In [ ]:
prop_nl_pts_empty = get_empty_pseudotrial_props(nl_pts,sData)
sparsity_dict['prop_nl_pts_empty']=prop_nl_pts_empty

In [ ]:
prop_nr_pts_empty = get_empty_pseudotrial_props(nr_pts,sData)
sparsity_dict['prop_nr_pts_empty']=prop_nr_pts_empty

In [ ]:
prop_pl_pts_empty = get_empty_pseudotrial_props(pl_pts,sData)
sparsity_dict['prop_pl_pts_empty']=prop_pl_pts_empty

In [ ]:
prop_pr_pts_empty = get_empty_pseudotrial_props(pr_pts,sData)
sparsity_dict['prop_pr_pts_empty']=prop_pr_pts_empty

In [ ]:
prop_l_e1_pts_empty = get_empty_pseudotrial_props(l_e1_pts,sData)
sparsity_dict['prop_l_e1_pts_empty']=prop_l_e1_pts_empty

In [ ]:
prop_r_e1_pts_empty = get_empty_pseudotrial_props(r_e1_pts,sData)
sparsity_dict['prop_r_e1_pts_empty']=prop_r_e1_pts_empty

In [ ]:
#save
sparsity_metrics_path = os.path.join(poppath, 'neural_event_sparsity_metrics.pkl')

with open(sparsity_metrics_path, 'wb') as f:
    pickle.dump(sparsity_dict,f)

In [ ]:
#using 25% as an arbitrary cutoff for 'too sparse'
if all(p <= 0.25 for p in [prop_nl_pts_empty, prop_nr_pts_empty, prop_pr_pts_empty, prop_pl_pts_empty]):
    print('fewer than 25% of pseudotrials in each identity/location subclass have no events')
else:
    print('at least one identity/location subclass has more than 25% of pseudotrials without any events')

In [ ]:
if all(p <= 0.25 for p in [prop_l_e1_pts_empty, prop_r_e1_pts_empty]):
    print('fewer than 25% of pseudotrials from each cup in the first empty trial have no events')
else:
    print('at least one cup in the first empty trial has more than 25% of pseudotrials occur without any events')

Get the continuous traces and mean population vectors for each defined pseudotrial

In [ ]:
nr_pt_dff, nr_pt_pvs = utils.traces_pv_by_bout(nr_pts,dff_Z)
nl_pt_dff, nl_pt_pvs = utils.traces_pv_by_bout(nl_pts,dff_Z)
pr_pt_dff, pr_pt_pvs = utils.traces_pv_by_bout(pr_pts,dff_Z)
pl_pt_dff, pl_pt_pvs = utils.traces_pv_by_bout(pl_pts,dff_Z)

l_e1_pt_dff, l_e1_pt_pvs = utils.traces_pv_by_bout(l_e1_pts,dff_Z)
r_e1_pt_dff, r_e1_pt_pvs = utils.traces_pv_by_bout(r_e1_pts,dff_Z)

----------

### Linear decoding of social identity and cup

In [ ]:
poppath

In [ ]:
#what are your minimum cell counts for comparisons across sessions and across animals?
min_neurons_within_animal = 86
min_neurons_across_animals = 33 #8020

In [ ]:
discontinuity_frames = []
for start, end in dropped_frame_info['dropped_ranges']:
    disc_frame = int(end)
    disc_frame += 1 #add 1 to convert back into a valid column number
    discontinuity_frames.append(disc_frame)

print(f"Found {len(discontinuity_frames)} discontinuities ending immediately before the following scope frames: {discontinuity_frames}")

In [ ]:
#decide whether to save out results or not
save_decoding_res = True

In [ ]:
def decoding_loop(comparison_label,
                  subclass_pts_dict,
                  class_assignments,
                  neural_data=None,
                  test_fraction=0.5,
                  min_train_per_class=2,
                  n_iterations_true=100,
                  n_iterations_null=30,
                  n_permutations_null=500,
                  discontinuity_frames=None,
                  random_seed=None,
                  C=1.0,
                  n_neurons_subsample=None,
                  null_shift=True,
                  null_shift_aware=True,
                  null_shuffle=True,
                  shuffle_level='interaction'):
    '''
    Perform decoding for true results and multiple null comparisons for one input dataset.
    '''
    if neural_data is None:
        raise ValueError("neural_data must be provided")
    
    res_dict = {}

    # Save out the number of pseudotrials that went in
    for key, value in subclass_pts_dict.items(): 
        res_dict[f'num_pts_{key}'] = len(value)

    # Save out information re whether you subsampled the population or not
    if n_neurons_subsample is not None:
        res_dict['subsampled_neurons'] = 'yes'
        res_dict['n_neurons_subsampled'] = n_neurons_subsample
    else:
        res_dict['subsampled_neurons'] = 'no'
        res_dict['n_neurons_subsampled'] = neural_data.shape[0]
        
    true_results = decoding.population_decoder(
        subclass_pts_dict=subclass_pts_dict,
        class_assignments=class_assignments,
        neural_data=neural_data,
        n_iterations=n_iterations_true,
        test_fraction=test_fraction,
        min_train_per_class=min_train_per_class,
        random_seed=random_seed,
        C=C,
        n_neurons_subsample=n_neurons_subsample        
    )
    print(f"{comparison_label} true decoding results: {true_results['mean_accuracy']:.3f} ± {true_results['std_accuracy']:.3f}")
    res_dict['true_decoding_results'] = true_results

    if n_neurons_subsample is None:
        decoding_importance, mean_weights = decoding.compute_decoding_importance(true_results)
        res_dict['decoding_weights'] = decoding_importance
        res_dict['signed_decoding_weights'] = mean_weights
    
    if null_shift:
        print("============")
        print("starting rotated permutations (not skipping discontinuities) for null distribution")
        null_shift_results = decoding.generate_null_distribution(
            subclass_pts_dict=subclass_pts_dict,
            class_assignments=class_assignments,
            neural_data=neural_data,
            n_permutations=n_permutations_null,
            n_decoder_iterations=n_iterations_null,
            null_method='circular',
            discontinuity_frames=discontinuity_frames,
            test_fraction=test_fraction,
            min_train_per_class=min_train_per_class,
            random_seed=random_seed,
            C=C,
            verbose=True,
            n_neurons_subsample=n_neurons_subsample
        )
    
        p_val_shift_accuracy = decoding.compute_p_value(true_results['mean_accuracy'], null_shift_results['null_accuracies'])
        p_val_shift_f1 = decoding.compute_p_value(true_results['mean_f1'], null_shift_results['null_f1s'])
        p_val_shift_auroc = decoding.compute_p_value(true_results['mean_auroc'], null_shift_results['null_aurocs'])
        print(f"{comparison_label} rotated null distribution, without skipping discontinuities: mean accuracy = {null_shift_results['mean_null_accuracy']:.3f}, p-value = {p_val_shift_accuracy:.3f}, mean F1 = {null_shift_results['mean_null_f1']:.3f}, p-value = {p_val_shift_f1:.3f}, mean AUROC = {null_shift_results['mean_null_auroc']:.3f}, p-value = {p_val_shift_auroc:.3f}")
        res_dict['null_shift_results'] = null_shift_results
        res_dict['p_val_shift_accuracy'] = p_val_shift_accuracy
        res_dict['p_val_shift_f1'] = p_val_shift_f1
        res_dict['p_val_shift_auroc'] = p_val_shift_auroc

    if null_shift_aware:
        print("============")
        print("starting rotated permutations (skipping discontinuities) for null distribution")
        null_shift_aware_results = decoding.generate_null_distribution(
            subclass_pts_dict=subclass_pts_dict,
            class_assignments=class_assignments,
            neural_data=neural_data,
            n_permutations=n_permutations_null,
            n_decoder_iterations=n_iterations_null,
            null_method='circular_aware',
            discontinuity_frames=discontinuity_frames,
            test_fraction=test_fraction,
            min_train_per_class=min_train_per_class,
            random_seed=random_seed,
            C=C,
            verbose=True,
            n_neurons_subsample=n_neurons_subsample
        )

        p_val_shift_aware_accuracy = decoding.compute_p_value(true_results['mean_accuracy'], null_shift_aware_results['null_accuracies'])
        p_val_shift_aware_f1 = decoding.compute_p_value(true_results['mean_f1'], null_shift_aware_results['null_f1s'])
        p_val_shift_aware_auroc = decoding.compute_p_value(true_results['mean_auroc'], null_shift_aware_results['null_aurocs'])
        print(f"{comparison_label} rotated null distribution, while skipping discontinuities: mean accuracy = {null_shift_aware_results['mean_null_accuracy']:.3f}, p-value = {p_val_shift_aware_accuracy:.3f}, mean F1 = {null_shift_aware_results['mean_null_f1']:.3f}, p-value = {p_val_shift_aware_f1:.3f}, mean AUROC = {null_shift_aware_results['mean_null_auroc']:.3f}, p-value = {p_val_shift_aware_auroc:.3f}")
        res_dict['null_shift_aware_results'] = null_shift_aware_results
        res_dict['p_val_shift_aware_accuracy'] = p_val_shift_aware_accuracy
        res_dict['p_val_shift_aware_f1'] = p_val_shift_aware_f1
        res_dict['p_val_shift_aware_auroc'] = p_val_shift_aware_auroc

    if null_shuffle:
        print("============")
        print("starting shuffled permutations for null distribution")
        null_shuffle_results = decoding.generate_null_label_shuffle(
            subclass_pts_dict=subclass_pts_dict,
            class_assignments=class_assignments,
            neural_data=neural_data,
            n_permutations=n_permutations_null,
            n_decoder_iterations=n_iterations_null,
            test_fraction=test_fraction,
            min_train_per_class=min_train_per_class,
            shuffle_level=shuffle_level,
            random_seed=random_seed,
            verbose=True,
            C=C,
            n_neurons_subsample=n_neurons_subsample
        )

        p_val_shuffle_accuracy = decoding.compute_p_value(true_results['mean_accuracy'], null_shuffle_results['null_accuracies'])
        p_val_shuffle_f1 = decoding.compute_p_value(true_results['mean_f1'], null_shuffle_results['null_f1s'])
        p_val_shuffle_auroc = decoding.compute_p_value(true_results['mean_auroc'], null_shuffle_results['null_aurocs'])
        print(f"{comparison_label} shuffled distribution (level={shuffle_level}): mean accuracy = {null_shuffle_results['mean_null_accuracy']:.3f}, p-value = {p_val_shuffle_accuracy:.3f}, mean F1 = {null_shuffle_results['mean_null_f1']:.3f}, p-value = {p_val_shuffle_f1:.3f}, mean AUROC = {null_shuffle_results['mean_null_auroc']:.3f}, p-value = {p_val_shuffle_auroc:.3f}")
        res_dict['null_shuffle_results'] = null_shuffle_results
        res_dict['p_val_shuffle_accuracy'] = p_val_shuffle_accuracy
        res_dict['p_val_shuffle_f1'] = p_val_shuffle_f1
        res_dict['p_val_shuffle_auroc'] = p_val_shuffle_auroc
    
    return res_dict

Decoding location from the first empty trial

In [ ]:
empty_location_subclass_pts_dict = {'l_e1': l_e1_pts, 'r_e1': r_e1_pts} #subclass: subclass's pseudotrials df
empty_location_class_assignments = {'left': ['l_e1'], 'right': ['r_e1']} #class 0: [subclasses], #class 1: [subclasses]

In [ ]:
empty_location_decoding_res = decoding_loop(comparison_label="location (empty trial 1)",
                                            subclass_pts_dict=empty_location_subclass_pts_dict,
                                            class_assignments=empty_location_class_assignments,
                                            neural_data=dff_Z,
                                            discontinuity_frames=discontinuity_frames,
                                            null_shift=False,
                                            null_shift_aware=True,
                                            null_shuffle=False)

In [ ]:
location_empty1_decoding_res_path = os.path.join(poppath, 'location_empty1_decoding_res.pkl')

if save_decoding_res == True:
    with open(location_empty1_decoding_res_path, 'wb') as f:
        pickle.dump(empty_location_decoding_res,f)

In [ ]:
empty_location_decoding_res_within_animal = decoding_loop(comparison_label="location (empty trial 1)",
                                            subclass_pts_dict=empty_location_subclass_pts_dict,
                                            class_assignments=empty_location_class_assignments,
                                            neural_data=dff_Z,
                                            discontinuity_frames=discontinuity_frames,
                                            n_neurons_subsample=min_neurons_within_animal,
                                            null_shift=False,
                                            null_shift_aware=False,
                                            null_shuffle=False)

In [ ]:
location_empty1_decoding_res_within_path = os.path.join(poppath, 'location_empty1_decoding_res_within_animal.pkl')

if save_decoding_res == True:
    with open(location_empty1_decoding_res_within_path, 'wb') as f:
        pickle.dump(empty_location_decoding_res_within_animal,f)

In [ ]:
empty_location_decoding_res_across_animals = decoding_loop(comparison_label="location (empty trial 1)",
                                            subclass_pts_dict=empty_location_subclass_pts_dict,
                                            class_assignments=empty_location_class_assignments,
                                            neural_data=dff_Z,
                                            discontinuity_frames=discontinuity_frames,
                                            n_neurons_subsample=min_neurons_across_animals,
                                            null_shift=False,
                                            null_shift_aware=False,
                                            null_shuffle=False)

In [ ]:
location_empty1_decoding_res_across_path = os.path.join(poppath, 'location_empty1_decoding_res_across_animals.pkl')

if save_decoding_res == True:
    with open(location_empty1_decoding_res_across_path, 'wb') as f:
        pickle.dump(empty_location_decoding_res_across_animals,f)

Decoding location from the social trials

In [ ]:
social_location_identity_subclass_pts_dict = {'p_r': pr_pts, 'p_l': pl_pts, 'n_r': nr_pts, 'n_l': nl_pts}

In [ ]:
social_location_class_assignments = {'right': ['p_r', 'n_r'], 'left': ['p_l', 'n_l']}

In [ ]:
social_location_decoding_res = decoding_loop(comparison_label="location (social_trials)",
                                            subclass_pts_dict=social_location_identity_subclass_pts_dict,
                                            class_assignments=social_location_class_assignments,
                                            neural_data=dff_Z,
                                            discontinuity_frames=discontinuity_frames,
                                            null_shift=False,
                                            null_shift_aware=True,
                                            null_shuffle=False)

In [ ]:
location_social_decoding_res_path = os.path.join(poppath, 'location_social_trials_decoding_res.pkl')

if save_decoding_res == True:
    with open(location_social_decoding_res_path, 'wb') as f:
        pickle.dump(social_location_decoding_res,f)

In [ ]:
social_location_decoding_res_within_animal = decoding_loop(comparison_label="location (social_trials)",
                                            subclass_pts_dict=social_location_identity_subclass_pts_dict,
                                            class_assignments=social_location_class_assignments,
                                            neural_data=dff_Z,
                                            discontinuity_frames=discontinuity_frames,
                                            n_neurons_subsample=min_neurons_within_animal,
                                            null_shift=False,
                                            null_shift_aware=False,
                                            null_shuffle=False)

In [ ]:
location_social_decoding_res_within_path = os.path.join(poppath, 'location_social_trials_decoding_res_within_animal.pkl')

if save_decoding_res == True:
    with open(location_social_decoding_res_within_path, 'wb') as f:
        pickle.dump(social_location_decoding_res_within_animal,f)

In [ ]:
social_location_decoding_res_across_animals = decoding_loop(comparison_label="location (social_trials)",
                                            subclass_pts_dict=social_location_identity_subclass_pts_dict,
                                            class_assignments=social_location_class_assignments,
                                            neural_data=dff_Z,
                                            discontinuity_frames=discontinuity_frames,
                                            n_neurons_subsample=min_neurons_across_animals,
                                            null_shift=False,
                                            null_shift_aware=False,
                                            null_shuffle=False)

In [ ]:
location_social_decoding_res_across_path = os.path.join(poppath, 'location_social_trials_decoding_res_across_animals.pkl')

if save_decoding_res == True:
    with open(location_social_decoding_res_across_path, 'wb') as f:
        pickle.dump(social_location_decoding_res_across_animals,f)

Decoding conspecific identity from the social trials

In [ ]:
identity_class_assignments = {'partner': ['p_r', 'p_l'], 'novel': ['n_r', 'n_l']}

In [ ]:
identity_decoding_res = decoding_loop(comparison_label="identity",
                                            subclass_pts_dict=social_location_identity_subclass_pts_dict,
                                            class_assignments=identity_class_assignments,
                                            neural_data=dff_Z,
                                            discontinuity_frames=discontinuity_frames,
                                            null_shift=False,
                                            null_shift_aware=True,
                                            null_shuffle=False)

In [ ]:
identity_decoding_res_path = os.path.join(poppath, 'identity_decoding_res.pkl')

if save_decoding_res == True:
    with open(identity_decoding_res_path, 'wb') as f:
        pickle.dump(identity_decoding_res,f)

In [ ]:
identity_decoding_res_within_animal = decoding_loop(comparison_label="identity",
                                            subclass_pts_dict=social_location_identity_subclass_pts_dict,
                                            class_assignments=identity_class_assignments,
                                            neural_data=dff_Z,
                                            discontinuity_frames=discontinuity_frames,
                                            n_neurons_subsample=min_neurons_within_animal,
                                            null_shift=False,
                                            null_shift_aware=False,
                                            null_shuffle=False)

In [ ]:
identity_decoding_res_within_path = os.path.join(poppath, 'identity_decoding_res_within_animal.pkl')

if save_decoding_res == True:
    with open(identity_decoding_res_within_path, 'wb') as f:
        pickle.dump(identity_decoding_res_within_animal,f)

In [ ]:
identity_decoding_res_across_animals = decoding_loop(comparison_label="identity",
                                            subclass_pts_dict=social_location_identity_subclass_pts_dict,
                                            class_assignments=identity_class_assignments,
                                            neural_data=dff_Z,
                                            discontinuity_frames=discontinuity_frames,
                                            n_neurons_subsample=min_neurons_across_animals,
                                            null_shift=False,
                                            null_shift_aware=False,
                                            null_shuffle=False)

In [ ]:
identity_decoding_res_across_path = os.path.join(poppath, 'identity_decoding_res_across_animals.pkl')

if save_decoding_res == True:
    with open(identity_decoding_res_across_path, 'wb') as f:
        pickle.dump(identity_decoding_res_across_animals,f)

--------------

### Mahalanobis distance ratio between different identities or different locations

1) Does population activity segregate for interactions with different identities (or different locations)?
2) If so, is this segregation stronger for identity or for location?
3) (Downstream code) how does affiliation or familiarity influence segregation?

Taking inspiration from the Remedios, Kennedy et al., 2017 Nature methodology

##### NEED AT LEAST 2 DISTINCT INTERACTIONS OF EACH SUBCLASS IN THE COMPARISON TO PERFORM THIS

In [ ]:
#using 1 sec pseudotrials because 1) not many are 'empty' and 2) staying consistent with decoding paradigm
#also, I'd like to balance identity across the location bins being used (or vice-versa) and this is esaier to achieve when I can flexibly add 1 second pseudotrials together 
#Grewe et al., 2017 uses 1 sec bins for their PVs for MD calculation as well

In [ ]:
#within a class comparison (partner, novel or left, right), we want to balance the presence of the other variable not being compared
#e.g., if partner interactions are mostly partner-left interactions, we could get an artificially low average MD from any given partner pseuedotrial to the 'partner distribution' if the population is actually mostly coding for left vs. right
#workflow:
#1) pick a single 1 second pseudotrial (will do this for all pseudotrials)
#2) drop all pseudotrials from the same interaction (to not perform comparisons to the same interaction)
#3) figure out which subclass for a given class has the fewest # of psuedotrials (e.g., partner left or partner right) and randomly subsample the larger class to the smaller class
#4) do this for both classes (i.e., partner and novel, or right and left)
#5) calculate mahalanobis distance between the chosen (test) vector and the distribution of vectors from all other psuedotrials of the same class (dsame) and the other class (dopp) and store differences
#6) repeat 2-5 20x to ensure cross-validation for all possible chosen pseudotrials and take average dsame and dopp
#7) repeat 1-6 for all pseudotrials
#8) take average dsame and dopp across all pseudotrials
#9) compute log(dopp/dsame) --> metric for how separated population activity is during interactions with partner vs. novel or left cup vs. right cup

map each subclass to its corresponding pseudotrials and their respective mean population vectors

In [ ]:
empty_pts_dict = {
    'l_e1':l_e1_pts,
    'r_e1':r_e1_pts
}

empty_pt_pvs_dict = {
    'l_e1':l_e1_pt_pvs,
    'r_e1':r_e1_pt_pvs
}

In [ ]:
social_pts_dict = {
    'pr':pr_pts,
    'pl':pl_pts,
    'nr':nr_pts,
    'nl':nl_pts
}

social_pt_pvs_dict = {
    'pr':pr_pt_pvs,
    'pl':pl_pt_pvs,
    'nr':nr_pt_pvs,
    'nl':nl_pt_pvs
}

Define Mahalanobis distance comparaison function and various helper functions

In [ ]:
def pt_pvs_dict_to_arr(pt_pvs_dict):
    '''generate a 2D array from a dictionary containing population vectors from different pseudotrials
    arg: pt_pvs_dict: a dictionary with values = mean population vector of each psuedotrial in the form of a 1D array (second output of utils.traces_pv_by_bout)
    returns: pt_pvs_arr = a 2D array with shape (n_neurons, n_pseudotrials)'''
    
    pvs = []
    for idx, arr in pt_pvs_dict.items():
        pvs.append(arr)
    pt_pvs_arr = np.column_stack(pvs)

    return pt_pvs_arr

In [ ]:
from sklearn.covariance import LedoitWolf
from scipy.spatial.distance import mahalanobis


def compute_shrinkage_covariance(data):
    """
    Compute covariance matrix using Ledoit-Wolf shrinkage.
    
    Parameters:
    -----------
    data : array-like, shape (n_neurons, n_samples)
        Population vectors as columns.
    
    Returns:
    --------
    covariance : ndarray, shape (n_neurons, n_neurons)
    """
    lw = LedoitWolf(store_precision=False)
    return lw.fit(data.T).covariance_

def regularize_covariance(cov_mat):
    epsilon = 1e-6
    cov_matrix_reg = cov_mat + (np.eye(cov_mat.shape[0]) * epsilon)
    return cov_matrix_reg

def mahalanobis_distance_comparison_four_subclasses(pts_dict, pt_pvs_dict, neural_data,
                                     comparison='identity', cov_method = 'lw', iterations_per_pv=20):
    """
    Compute Mahalanobis distance-based separation metric for population activity
    between two classes of interactions split into four subclasses.
    
    For each pseudotrial (focal vector), computes distance to the same-class
    distribution (d_same) and opposite-class distribution (d_opp), each using
    its own Ledoit-Wolf regularized covariance matrix. Within-class subclass
    balancing controls for confounding variables (e.g., balancing left/right
    within partner when comparing identity).
    
    Parameters:
    -----------
    pts_dict : dict
        Keys are subclass labels ('pr', 'pl', 'nr', 'nl'), values are DataFrames
        of pseudotrials with columns including 'start', 'end', 'int'.
    pt_pvs_dict : dict
        Keys match pts_dict. Values are arrays of shape (n_neurons, n_pseudotrials)
        from extract_population_vectors.
    neural_data : pd.DataFrame
        Z-scored dF/F or similar, shape (n_neurons, n_frames). Used to recompute
        population vectors after subsampling.
    comparison : str
        'identity' (partner vs novel) or 'location' (right vs left)
    cov_method: str
        Which method to apply to the covariance matrix to prevent singularity/non-invertible covariance matrices.
        Options are 'lw' = Ledoit-Wolf shrinkage, 'diag_reg' = regularization by adding a small value to the diagonal, or 'none' which does nothing to the covariance matrix
        ('none' will likely fail if the # of neurons is greater than the # of pseudotrials, see https://stats.stackexchange.com/questions/49826/what-to-do-when-sample-covariance-matrix-is-not-invertible?rq=1,
        https://stats.stackexchange.com/questions/37743/singular-covariance-matrix-in-mahalanobis-distance-in-matlab#comment74099_37744,
        https://stats.stackexchange.com/questions/218619/mahalanobis-distance-what-if-s-is-not-invertible,
        https://en.wikipedia.org/wiki/Invertible_matrix,
        Ledoit and Wolf, 2001 Journal of Multivariate Analysis,
        and Biane et al., 2025 Neuron for application of Ledoit-Wolf in Mahalanobis distance in calcium imaging data)
    iterations_per_pv : int
        Number of random subsample iterations per focal pseudotrial (default=20).
    
    Returns:
    --------
    same_mds : list
        Mean d_same for each pseudotrial (averaged over iterations).
    diff_mds : list
        Mean d_opp for each pseudotrial (averaged over iterations).
    md_ratio : float
        log(mean_d_opp / mean_d_same). Positive = good separation.
    """

    if comparison == 'identity':
        class0 = ['pr', 'pl']  # partner
        class1 = ['nr', 'nl']  # novel
    elif comparison == 'location':
        class0 = ['pr', 'nr']  # right
        class1 = ['pl', 'nl']  # left
    else:
        raise ValueError(f"comparison must be 'identity' or 'location', got '{comparison}'")

    all_subclasses = class0 + class1

    same_mds = []
    diff_mds = []

    for subclass in all_subclasses:
        # Determine same-class and diff-class groupings for this subclass
        if subclass in class0:
            same = class0
            diff = class1
        else:
            same = class1
            diff = class0

        # Get the pseudotrials and population vectors for this subclass
        subclass_pts = pts_dict[subclass].copy()
        subclass_pt_pvs = pt_pvs_dict[subclass]
        subclass_pt_pvs_arr = pt_pvs_dict_to_arr(subclass_pt_pvs)
        
        # Identify the other subclass in the same class
        other_same_subclass = [x for x in same if x != subclass][0]

        # Loop through each pseudotrial as the focal vector
        for idx in range(subclass_pts.shape[0]):
            # Get the focal population vector (a single column)
            focal_pv = subclass_pt_pvs_arr[:, idx]

            # Get the interaction this pseudotrial came from
            focal_interaction = subclass_pts.iloc[idx]['int']

            # Drop all pseudotrials from the same interaction within this subclass
            subclass_pts_other_ints = subclass_pts[subclass_pts['int'] != focal_interaction].copy()

            # Store originals for the other subclasses (not affected by interaction exclusion)
            other_same_pts_orig = pts_dict[other_same_subclass].copy()

            #get the pseudotrials from the two subclasses in the other class
            diff_subclass0, diff_subclass1 = diff[0], diff[1]
            diff0_pts_orig = pts_dict[diff_subclass0].copy()
            diff1_pts_orig = pts_dict[diff_subclass1].copy()

            # Store per-iteration distances for this focal pseudotrial
            pt_same_mds = []
            pt_diff_mds = []

            for n in range(iterations_per_pv):
                # --- Balance within same class ---
                same_sub_pts = subclass_pts_other_ints.copy()
                other_same_pts = other_same_pts_orig.copy()

                min_same = min(len(same_sub_pts), len(other_same_pts))
                if min_same == 0:
                    continue

                if len(same_sub_pts) > min_same:
                    same_sub_pts = utils.sample_pseudotrials_balanced(same_sub_pts, min_same)
                if len(other_same_pts) > min_same:
                    other_same_pts = utils.sample_pseudotrials_balanced(other_same_pts, min_same)

                # Compute population vectors for balanced same-class pseudotrials
                _, same_sub_pvs = utils.traces_pv_by_bout(same_sub_pts, neural_data)
                _, other_same_pvs = utils.traces_pv_by_bout(other_same_pts, neural_data)

                same_sub_pvs_arr = pt_pvs_dict_to_arr(same_sub_pvs)
                other_same_pvs_arr = pt_pvs_dict_to_arr(other_same_pvs)
                same_pvs = np.hstack([same_sub_pvs_arr, other_same_pvs_arr])  # (n_neurons, n_same_pts)

                #generate the same-class centroid and covariance matrix (using the function specified in the input argument)
                same_centroid = same_pvs.mean(axis=1)
                
                if cov_method=='lw':
                    same_cov = compute_shrinkage_covariance(same_pvs)
                elif cov_method=='diag_reg':
                    same_cov_before_reg = np.cov(same_pvs)
                    same_cov = regularize_covariance(same_cov_before_reg)
                elif cov_method=='none': #just calculate the covariance matrix without performing any additional calculations
                    same_cov = np.cov(same_pvs)
                else:
                    raise ValueError(f"cov_method must be one of 'lw', 'diag_reg', or 'none'; got '{cov_method}'")

                #now invert the covariance matrix
                same_cov_inv = np.linalg.inv(same_cov)

                # --- Balance within diff class ---
                diff0_pts = diff0_pts_orig.copy()
                diff1_pts = diff1_pts_orig.copy()

                min_diff = min(len(diff0_pts), len(diff1_pts))
                if min_diff == 0:
                    continue

                if len(diff0_pts) > min_diff:
                    diff0_pts = utils.sample_pseudotrials_balanced(diff0_pts, min_diff)
                if len(diff1_pts) > min_diff:
                    diff1_pts = utils.sample_pseudotrials_balanced(diff1_pts, min_diff)

                # Compute population vectors for balanced diff-class pseudotrials
                _, diff0_pvs = utils.traces_pv_by_bout(diff0_pts, neural_data)
                _, diff1_pvs = utils.traces_pv_by_bout(diff1_pts, neural_data)

                diff0_pvs_arr = pt_pvs_dict_to_arr(diff0_pvs)
                diff1_pvs_arr = pt_pvs_dict_to_arr(diff1_pvs)
                diff_pvs = np.hstack([diff0_pvs_arr, diff1_pvs_arr])  # (n_neurons, n_diff_pts)
                
                # Diff-class centroid and covariance
                diff_centroid = diff_pvs.mean(axis=1)

                if cov_method=='lw':
                    diff_cov = compute_shrinkage_covariance(diff_pvs)
                elif cov_method=='diag_reg':
                    diff_cov_before_reg = np.cov(diff_pvs)
                    diff_cov = regularize_covariance(diff_cov_before_reg)
                elif cov_method=='none': #just calculate the covariance matrix without performing any additional calculations
                    diff_cov = np.cov(diff_pvs)
                
                diff_cov_inv = np.linalg.inv(diff_cov)

                # --- Compute Mahalanobis distances ---
                md_same = mahalanobis(focal_pv, same_centroid, same_cov_inv)
                md_diff = mahalanobis(focal_pv, diff_centroid, diff_cov_inv)

                pt_same_mds.append(md_same)
                pt_diff_mds.append(md_diff)

            # Average across iterations for this pseudotrial
            if len(pt_same_mds) > 0:
                same_mds.append(np.mean(pt_same_mds))
                diff_mds.append(np.mean(pt_diff_mds))

    # Compute summary metric
    mean_same = np.mean(same_mds)
    mean_diff = np.mean(diff_mds)

    # Positive = focal vectors are closer to own class than opposite class
    md_ratio = np.log(mean_diff / mean_same)

    return same_mds, diff_mds, md_ratio

In [ ]:
def plot_pseudotrials_mahalanobis_distances(same_mds, diff_mds, md_ratio, comparison='identity',
                                            binsize=30, figsize=(6, 4), save_path=None):
    fig, ax = plt.subplots(figsize=figsize)
    bins = np.histogram_bin_edges(same_mds + diff_mds, bins=binsize)
    ax.hist(same_mds, bins=bins, alpha=0.5, label='d_same (to own class)', color='steelblue')
    ax.hist(diff_mds, bins=bins, alpha=0.5, label='d_diff (to opposite class)', color='coral')
    ax.axvline(np.mean(same_mds), color='steelblue', linestyle='--', linewidth=1.5)
    ax.axvline(np.mean(diff_mds), color='coral', linestyle='--', linewidth=1.5)
    ax.set_xlabel('Mahalanobis Distance')
    ax.set_ylabel('Count')
    ax.set_title(f'{comparison.capitalize()}: log(d_diff/d_same) = {md_ratio:.3f}')
    ax.legend()
    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=300, bbox_inches='tight')


def plot_pseudotrials_mahalanobis_cdf(same_mds, diff_mds, md_ratio, comparison='identity',
                                      figsize=(6, 4), save_path=None):
    fig, ax = plt.subplots(figsize=figsize)

    same_sorted = np.sort(same_mds)
    diff_sorted = np.sort(diff_mds)
    same_cdf = np.arange(1, len(same_sorted) + 1) / len(same_sorted)
    diff_cdf = np.arange(1, len(diff_sorted) + 1) / len(diff_sorted)

    ax.step(same_sorted, same_cdf, where='post', label='d_same (to own class)', color='steelblue')
    ax.step(diff_sorted, diff_cdf, where='post', label='d_diff (to opposite class)', color='coral')
    ax.axvline(np.mean(same_mds), color='steelblue', linestyle='--', linewidth=1.5, alpha=0.7)
    ax.axvline(np.mean(diff_mds), color='coral', linestyle='--', linewidth=1.5, alpha=0.7)
    ax.set_xlabel('Mahalanobis Distance')
    ax.set_ylabel('Cumulative Probability')
    ax.set_title(f'{comparison.capitalize()}: log(d_diff/d_same) = {md_ratio:.3f}')
    ax.legend()
    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=300, bbox_inches='tight')

Initialize dictionary to save results into

In [ ]:
md_dict = {}

In [ ]:
save_figs = True

Option to run distance comparison with two different handlings of the covariance matrix to make the matrix invertible - Ledoit-Wolf shrinkage and regularization with adding a small value to the diagonal

Ledoit-Wolf:
1. Social identity

In [ ]:
#run with ledoit wolf covariance matrix shrinkage (my 'default' approach)
same_mds_identity_lw, diff_mds_identity_lw, md_ratio_identity_lw = mahalanobis_distance_comparison_four_subclasses(social_pts_dict, social_pt_pvs_dict, dff_Z,
                                     comparison='identity', cov_method = 'lw', iterations_per_pv=20)

In [ ]:
md_dict['mds_same_identity_lw'] = same_mds_identity_lw
md_dict['mds_diff_identity_lw'] = diff_mds_identity_lw
md_dict['md_ratio_identity_lw'] = md_ratio_identity_lw

In [ ]:
if save_figs == True:
    fig_path = os.path.join(poppath, "identity_md_lw_histogram.png")
else:
    fig_path = None

plot_pseudotrials_mahalanobis_distances(same_mds_identity_lw,diff_mds_identity_lw,md_ratio_identity_lw,comparison='identity (Ledoit-Wolf covariance)',
                                            binsize=30, figsize=(6, 4), save_path=fig_path)

In [ ]:
if save_figs == True:
    fig_path = os.path.join(poppath, "identity_md_lw_cumprob.png")
else:
    fig_path = None

plot_pseudotrials_mahalanobis_cdf(same_mds_identity_lw,diff_mds_identity_lw,md_ratio_identity_lw,comparison='identity (Ledoit-Wolf covariance)',
                                            figsize=(6, 4), save_path=fig_path)

2. cup location

In [ ]:
same_mds_location_lw, diff_mds_location_lw, md_ratio_location_lw = mahalanobis_distance_comparison_four_subclasses(social_pts_dict, social_pt_pvs_dict, dff_Z,
                                     comparison='location', cov_method = 'lw', iterations_per_pv=20)

In [ ]:
md_dict['mds_same_location_lw'] = same_mds_location_lw
md_dict['mds_diff_location_lw'] = diff_mds_location_lw
md_dict['md_ratio_location_lw'] = md_ratio_location_lw

In [ ]:
if save_figs == True:
    fig_path = os.path.join(poppath, "location_md_lw_histogram.png")
else:
    fig_path = None

plot_pseudotrials_mahalanobis_distances(same_mds_location_lw,diff_mds_location_lw,md_ratio_location_lw,comparison='location (Ledoit-Wolf covariance)',
                                            binsize=30, figsize=(6, 4), save_path=fig_path)

In [ ]:
if save_figs == True:
    fig_path = os.path.join(poppath, "location_md_lw_cumprob.png")
else:
    fig_path = None

plot_pseudotrials_mahalanobis_cdf(same_mds_location_lw,diff_mds_location_lw,md_ratio_location_lw,comparison='location (Ledoit-Wolf covariance)',
                                            figsize=(6, 4), save_path=fig_path)

Regularization via adding a small number to the diagonal
1. Social identity

In [ ]:
#run with regularization of the diagonal
same_mds_identity_dr, diff_mds_identity_dr, md_ratio_identity_dr = mahalanobis_distance_comparison_four_subclasses(social_pts_dict, social_pt_pvs_dict, dff_Z,
                                     comparison='identity', cov_method = 'diag_reg', iterations_per_pv=20)

In [ ]:
md_dict['mds_same_identity_dr'] = same_mds_identity_dr
md_dict['mds_diff_identity_dr'] = diff_mds_identity_dr
md_dict['md_ratio_identity_dr'] = md_ratio_identity_dr

In [ ]:
if save_figs == True:
    fig_path = os.path.join(poppath, "identity_md_diagreg_histogram.png")
else:
    fig_path = None

plot_pseudotrials_mahalanobis_distances(same_mds_identity_dr,diff_mds_identity_dr,md_ratio_identity_dr,comparison='identity (diagonal regul.)',
                                            binsize=30, figsize=(6, 4), save_path=fig_path)

In [ ]:
if save_figs == True:
    fig_path = os.path.join(poppath, "identity_md_diagreg_cumprob.png")
else:
    fig_path = None

plot_pseudotrials_mahalanobis_cdf(same_mds_identity_dr,diff_mds_identity_dr,md_ratio_identity_dr,comparison='identity (diagonal regul.)',
                                            figsize=(6, 4), save_path=fig_path)

In [ ]:
act_rints_merged_empty1

In [ ]:
pr_ints_merged

2. cup location

In [ ]:
same_mds_location_dr, diff_mds_location_dr, md_ratio_location_dr = mahalanobis_distance_comparison_four_subclasses(social_pts_dict, social_pt_pvs_dict, dff_Z,
                                     comparison='location', cov_method = 'diag_reg', iterations_per_pv=20)

In [ ]:
md_dict['mds_same_location_dr'] = same_mds_location_dr
md_dict['mds_diff_location_dr'] = diff_mds_location_dr
md_dict['md_ratio_location_dr'] = md_ratio_location_dr

In [ ]:
if save_figs == True:
    fig_path = os.path.join(poppath, "location_md_diagreg_histogram.png")
else:
    fig_path = None

plot_pseudotrials_mahalanobis_distances(same_mds_location_dr,diff_mds_location_dr,md_ratio_location_dr,comparison='location (diagonal regul.)',
                                            binsize=30, figsize=(6, 4), save_path=fig_path)

In [ ]:
if save_figs == True:
    fig_path = os.path.join(poppath, "location_md_diagreg_cumprob.png")
else:
    fig_path = None

plot_pseudotrials_mahalanobis_cdf(same_mds_location_dr,diff_mds_location_dr,md_ratio_location_dr,comparison='location (diagonal regul.)',
                                            figsize=(6, 4), save_path=fig_path)

In [ ]:
print(f" MD ratio location (ledoit-wolf): {md_dict['md_ratio_location_lw']}")
print(f" MD ratio identity (ledoit-wolf): {md_dict['md_ratio_identity_lw']}")
print("")
print(f" MD ratio location (diagonal regularization): {md_dict['md_ratio_location_dr']}")
print(f" MD ratio identity (diagonal regularization): {md_dict['md_ratio_identity_dr']}")

Now compute the same Mahalanobis distance comparison but for the data from the first empty trial

Because there are only two subclasses (left and right) we don't need to subsample to balance another variable within classes. So I wrote a different (but very similar) function to handle this

In [ ]:
def mahalanobis_distance_comparison_two_subclasses(pts_dict, pt_pvs_dict, neural_data,
                                                   cov_method = 'lw'):
    """
    Compute Mahalanobis distance-based separation metric for population activity
    between two classes of interactions, not split into subclasses.
    
    For each pseudotrial (focal vector), computes distance to the same-class
    distribution (d_same) and opposite-class distribution (d_opp), each using
    its own Ledoit-Wolf regularized covariance matrix.
    
    Parameters: same as for the four_subclasses function but without a 'comparison' term (there is only one possible comparison between the two subclasses)
    
    Returns: same as for the four_subclasses function:
    same_mds : list
        Mean d_same for each pseudotrial (averaged over iterations).
    diff_mds : list
        Mean d_opp for each pseudotrial (averaged over iterations).
    md_ratio : float
        log(mean_d_opp / mean_d_same). Positive = good separation.
    """    
    
    same_mds = []
    diff_mds = []

    subclasses = list(pts_dict.keys())

    for subclass in subclasses:
        same = subclass
        diff = [x for x in subclasses if x != subclass][0]
        
        same_pts = pts_dict[same].copy()
        same_pt_pvs = pt_pvs_dict[same]
        same_pt_pvs_arr = pt_pvs_dict_to_arr(same_pt_pvs)

        diff_pts = pts_dict[diff].copy()
        diff_pt_pvs = pt_pvs_dict[diff]
        diff_pt_pvs_arr = pt_pvs_dict_to_arr(diff_pt_pvs)
        
         # Diff-class centroid and covariance
        diff_centroid = diff_pt_pvs_arr.mean(axis=1)

        if cov_method=='lw':
            diff_cov = compute_shrinkage_covariance(diff_pt_pvs_arr)
        elif cov_method=='diag_reg':
            diff_cov_before_reg = np.cov(diff_pt_pvs_arr)
            diff_cov = regularize_covariance(diff_cov_before_reg)
        elif cov_method=='none': #just calculate the covariance matrix without performing any additional calculations
            diff_cov = np.cov(diff_pt_pvs_arr)
                
        diff_cov_inv = np.linalg.inv(diff_cov)

        # Loop through each pseudotrial as the focal vector
        for idx in range(same_pts.shape[0]):
            # Get the focal population vector (a single column)
            focal_pv = same_pt_pvs_arr[:, idx]

            # Get the interaction this pseudotrial came from
            focal_interaction = same_pts.iloc[idx]['int']

            # Drop all pseudotrials from the same interaction within this subclass
            same_pts_other_ints = same_pts[same_pts['int'] != focal_interaction].copy()
            
            # Compute population vectors for the same class pseudotrials from different interactions as the focal PV
            _, same_other_ints_pvs = utils.traces_pv_by_bout(same_pts_other_ints, neural_data)

            same_other_ints_pvs_arr = pt_pvs_dict_to_arr(same_other_ints_pvs) # (n_neurons, n_same_pts)

            #generate the same-class centroid and covariance matrix (using the function specified in the input argument)
            same_centroid = same_other_ints_pvs_arr.mean(axis=1)
                
            if cov_method=='lw':
                same_cov = compute_shrinkage_covariance(same_other_ints_pvs_arr)
            elif cov_method=='diag_reg':
                same_cov_before_reg = np.cov(same_other_ints_pvs_arr)
                same_cov = regularize_covariance(same_cov_before_reg)
            elif cov_method=='none': #just calculate the covariance matrix without performing any additional calculations
                same_cov = np.cov(same_other_ints_pvs_arr)
            else:
                raise ValueError(f"cov_method must be one of 'lw', 'diag_reg', or 'none'; got '{cov_method}'")

            #now invert the covariance matrix
            same_cov_inv = np.linalg.inv(same_cov)

            # --- Compute Mahalanobis distances ---
            md_same = mahalanobis(focal_pv, same_centroid, same_cov_inv)
            md_diff = mahalanobis(focal_pv, diff_centroid, diff_cov_inv)

            same_mds.append(md_same)
            diff_mds.append(md_diff)

    # Compute summary metric
    mean_same = np.mean(same_mds)
    mean_diff = np.mean(diff_mds)

    # Positive = focal vectors are closer to own class than opposite class
    md_ratio = np.log(mean_diff / mean_same)

    return same_mds, diff_mds, md_ratio

In [ ]:
same_mds_location_empty1_lw, diff_mds_location_empty1_lw, md_ratio_location_empty1_lw = mahalanobis_distance_comparison_two_subclasses(empty_pts_dict, empty_pt_pvs_dict,
                                                                                                                              dff_Z, cov_method='lw')

In [ ]:
poppath

In [ ]:
if save_figs == True:
    fig_path = os.path.join(poppath, "empty1_location_md_lw_histogram.png")
else:
    fig_path = None

plot_pseudotrials_mahalanobis_distances(same_mds_location_empty1_lw,diff_mds_location_empty1_lw,md_ratio_location_empty1_lw,comparison='location (empty cups) (ledoit-wolf covariance)',
                                            binsize=30, figsize=(6, 4), save_path=fig_path)

In [ ]:
if save_figs == True:
    fig_path = os.path.join(poppath, "empty1_location_md_lw_cumprob.png")
else:
    fig_path = None

plot_pseudotrials_mahalanobis_cdf(same_mds_location_empty1_lw,diff_mds_location_empty1_lw,md_ratio_location_empty1_lw,comparison='location (empty cups) (ledoit-wolf covariance)',
                                            figsize=(6, 4), save_path=fig_path)

In [ ]:
md_dict['mds_same_location_empty1_lw'] = same_mds_location_empty1_lw
md_dict['mds_diff_location_empty1_lw'] = diff_mds_location_empty1_lw
md_dict['md_ratio_location_empty1_lw'] = md_ratio_location_empty1_lw

In [ ]:
same_mds_location_empty1_dr, diff_mds_location_empty1_dr, md_ratio_location_empty1_dr = mahalanobis_distance_comparison_two_subclasses(empty_pts_dict, empty_pt_pvs_dict,
                                                                                                                              dff_Z, cov_method='diag_reg')

In [ ]:
if save_figs == True:
    fig_path = os.path.join(poppath, "empty1_location_md_diagreg_histogram.png")
else:
    fig_path = None

plot_pseudotrials_mahalanobis_distances(same_mds_location_empty1_dr,diff_mds_location_empty1_dr,md_ratio_location_empty1_dr,comparison='location (empty cups) (diagonal regul.)',
                                            binsize=30, figsize=(6, 4), save_path=fig_path)

In [ ]:
if save_figs == True:
    fig_path = os.path.join(poppath, "empty1_location_md_diagreg_cumprob.png")
else:
    fig_path = None

plot_pseudotrials_mahalanobis_cdf(same_mds_location_empty1_dr,diff_mds_location_empty1_dr,md_ratio_location_empty1_dr,comparison='location (empty cups) (diagonal regul.)',
                                            figsize=(6, 4), save_path=fig_path)

In [ ]:
md_dict['mds_same_location_empty1_dr'] = same_mds_location_empty1_dr
md_dict['mds_diff_location_empty1_dr'] = diff_mds_location_empty1_dr
md_dict['md_ratio_location_empty1_dr'] = md_ratio_location_empty1_dr

Save out results

In [ ]:
md_results_path = os.path.join(poppath, 'mahalanobis_distance_comparisons.pkl')

with open(md_results_path, 'wb') as f:
    pickle.dump(md_dict,f)